# AXIFEM element evidence

This result-bearing notebook records the current proof that the six production AXIFEM element paths run and are guarded by executable regression gates: P1, Q1, P2, Q2, P2 curved, and Q2 curved.

In [1]:
from pathlib import Path
import json

for candidate in [Path("axifem_element_evidence.json"), Path("docs/axifem/axifem_element_evidence.json")]:
    if candidate.exists():
        evidence = json.loads(candidate.read_text(encoding="utf-8"))
        break
else:
    raise FileNotFoundError("axifem_element_evidence.json")

print(json.dumps(evidence["version_stamp"], indent=2, ensure_ascii=False))


{
  "runtime_radia_version": "4.94.0",
  "executed_at_utc": "2026-06-26T02:02:08.730485+00:00",
  "python": "3.12.10",
  "platform": "Windows-2022Server-10.0.20348-SP0",
  "git_head": "53303f1a",
  "git_dirty": true
}


## Evidence matrix

Each row links one element path to the production selection API and the regression gate that would fail if the path regressed.

In [2]:
from IPython.display import Markdown, display
rows = evidence["evidence_matrix"]
headers = ["Element path", "Production selection", "Evidence gate", "What would fail if broken"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
for row in rows:
    lines.append("| " + " | ".join(row[h] for h in headers) + " |")
display(Markdown("\n".join(lines)))


| Element path | Production selection | Evidence gate | What would fail if broken |
| --- | --- | --- | --- |
| P1 triangle | H1Henrotte(mesh, order=1) on triangles | test_element_matrices.py + test_python_reference_consistency.py | singular/NaN element matrices, non-symmetric Mr/Mz, wrong const-phi null mode, Python reference drift |
| Q1 quad | H1Henrotte(mesh, order=1) on axis-aligned quads | test_q1_vdof.py + Q1 V-DOF reference in test_python_reference_consistency.py | uniform Bz field no longer reproduced; C++ stiffness/mass spectrum no longer matches Python V-DOF reference |
| P2 triangle | H1Henrotte(mesh, order=2) on triangles | test_p2_axis_eddy.py + test_python_reference_consistency.py | dead face-center DOFs become free, K/M lose full rank, P1/P2 slowest eddy mode diverges |
| Q2 quad | H1Henrotte(mesh, order=2, curvedquad=False) on axis-aligned quads | test_q2_curved.py straight-quad equivalence + Cauer Q2 stored validation summary | axis-aligned Q2 eigenvalue changes; Q2 Cauer time constants drift |
| P2 curved triangle | mesh.Curve(2) with H1Henrotte(mesh, order=2) | test_p2_curved_magsta.py | Curve(2) no longer improves sphere volume/total-flux error |
| Q2 curved quad | H1Henrotte(mesh, order=2, curvedquad=True) | test_q2_curved.py | curved Q2 no longer matches straight Q2 on rectangles or converges on annular skewed quads |

## Pytest proof

This is the verbose output from the element-gate subset. It is intentionally embedded so the notebook remains useful on GitHub without re-running locally.

In [3]:
print(" ".join(evidence["pytest"]["command"]))
print()
print(evidence["pytest"]["stdout"])


python -m pytest tests\axifem\test_element_matrices.py tests\axifem\test_python_reference_consistency.py tests\axifem\test_q1_vdof.py tests\axifem\test_p2_axis_eddy.py tests\axifem\test_p2_curved_magsta.py tests\axifem\test_q2_curved.py tests\axifem\test_taskmanager_race.py -q -s

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.0.3, pluggy-1.6.0
rootdir: W:\00_CAE\Radia\01_GitHub
configfile: pytest.ini (WARNING: ignoring pytest config in pyproject.toml!)
plugins: anyio-4.13.0, rerunfailures-16.3
collected 20 items

tests\axifem\test_element_matrices.py [OK] interior triangle Mr, Mz are symmetric
.[OK] flag=2 (2 nodes on axis) executes without NaN/Inf
.[OK] flag=1 (1 node on axis) executes without NaN/Inf
.[OK] interpolation: B_z=0.859437 B_r=-246.690162
.  Mr * (1/r_j) = [-7.80625564e-18  1.31838984e-16  0.00000000e+00]
  Mz * (1/r_j) = [-7.02563008e-17 -9.62771529e-17 -1.28369537e-16]
[OK] const phi (V-form: 1/

## Stored validation summaries

The Cauer disk summaries are pulled from the promoted axifem validation results under `validation_test/axifem/research/verification`. The reference numbers are stored regression references, not a live external solver call.

In [4]:
from IPython.display import Markdown, display
lines = [
    "| Family | Selected mesh | Mesh DOF/free | tau_pair_us first 6 | stored reference first 6 | rel(first) |",
    "| --- | --- | --- | --- | --- | --- |",
]
for row in evidence["cauer_validation_summary"]:
    mesh = row["mesh"]
    mesh_s = f"ne={mesh.get('ne')}, ndof={mesh.get('ndof')}, free={mesh.get('free')}"
    lines.append(
        f"| {row['family']} | {row['selected_mesh']} | {mesh_s} | {row['tau_pair_us_first6']} | {row['stored_bem_reference_us_first6']} | {row['first_tau_relative_to_reference']} |"
    )
display(Markdown("\n".join(lines)))


| Family | Selected mesh | Mesh DOF/free | tau_pair_us first 6 | stored reference first 6 | rel(first) |
| --- | --- | --- | --- | --- | --- |
| Q1 quad | very fine | ne=15170, ndof=15438, free=14904 | [218.052147, 77.773176, 39.374895, 23.140683, 16.064211, 13.011787] | [224.307059, 88.423956, 49.539868, 32.090204, 23.339598, 22.588437] | 0.027885 |
| Q2 quad | fine | ne=2530, ndof=10323, free=9919 | [218.707571, 78.121442, 39.54416, 23.157879, 16.073587, 13.118726] | [224.307059, 88.423956, 49.539868, 32.090204, 23.339598, 22.588437] | 0.024963 |

## Timing

The result JSON keeps a small timing breakdown. Heavy phases are listed first.

In [5]:
print(json.dumps(evidence["timing_breakdown_s"], indent=2, ensure_ascii=False))


{
  "pytest_axifem_element_subset": 1.08,
  "cauer_json_summary_load": 0.000668,
  "notebook_artifact_generation": 0.032237
}
